# Colab to generate data s2 cells given a kml file of the region.

Steps:

1.   Extract polygons from the given kml file.
2.   Obtain s2 cells that are within the polygons.

In [ ]:
import io
import json
import os
import fiona

In [ ]:
!pip install s2sphere
!pip install geopandas
import geopandas as gpd
import s2sphere as s2

In [ ]:
fiona.drvsupport.supported_drivers['KML'] = 'rw'

In [ ]:
def fetch_polygons_in_kml(kml_path: str) -> list[list[tuple[float, float]]]:
    """Fetches list of polygons for all geometries in kml file.

    Args:
      kml_path: path to kml file

    Returns:
      returns list of lat lng polygons for all geometries in kml file
    """
    gdf = gpd.read_file(kml_path, driver='KML')
    polygon_vertices = []
    for i in range(gdf.shape[0]):
        exterior_coords = []
        geometry = gdf.iloc[i].geometry
        if geometry.geom_type == 'MultiPolygon':
            # Iterate over each Polygon in the MultiPolygon
            for polygon in geometry.geoms:
                exterior_coords += list(polygon.exterior.coords)
        elif geometry.geom_type == 'Polygon':
            exterior_coords += list(geometry.exterior.coords)
        else:
            continue  # Skip non-polygon geometries

        polygon_vertices.append(
            [(float(coord[1]), float(coord[0])) for coord in exterior_coords]
        )
    return polygon_vertices


In [ ]:
def get_covering_s2_cells(polygon, level):
  """Gets a list of S2 cell IDs of a given level that cover the bounding box of a polygon.

  A visualization of this covering can be seen at go/s2covering.

  Args:
    lat: list of coordinates i.e. [lat, lng] of a polygon or the corner
      coordinates of a rectangle.
    level: level at which s2cells are to be generated, default level is 10 where
      a cell has edge length between 7 to 10km.

  Raises:
    Exception: no s2 cells are found i.e. if illegal polygon is passed.
  Returns:
    s2_cover: list of s2cell ids(ints) tht cover the whole polygon.
  """

  lat_lngs = [s2.LatLng.from_degrees(lat, lng) for lat, lng in polygon]

  lats = [l.lat().radians for l in lat_lngs]
  lngs = [l.lng().radians for l in lat_lngs]

  min_lat = min(lats)
  max_lat = max(lats)
  min_lng = min(lngs)
  max_lng = max(lngs)

  s2_min_pt = s2.LatLng.from_radians(min_lat, min_lng)
  s2_max_pt = s2.LatLng.from_radians(max_lat, max_lng)
  rect = s2.LatLngRect.from_point_pair(s2_min_pt, s2_max_pt)

  coverer = s2.RegionCoverer()
  coverer.min_level = level
  coverer.max_level = level
  covering = coverer.get_covering(rect)
  if not covering:
    print('No s2 cover for polygon %s', polygon)
  s2_cell_ids = [cell.id() for cell in covering]

  return s2_cell_ids


def get_s2_cells_covering_kml(polygons_vertices: list[list[tuple[float, float]]], s2_cell_level: int) -> set[str]:
  """

  Args:
    polygons_vertices: list of polygons
    s2_cell_level: s2 cell level

  Returns: s2 cells at specified level for a region

  """
  s2_cell_ids_covering_kml = set()
  for polygon in polygons_vertices:
    s2_cell_ids_covering_polygon = get_covering_s2_cells(
        polygon=polygon, level=s2_cell_level
    )
    s2_cell_ids_covering_kml.update(s2_cell_ids_covering_polygon)
  return s2_cell_ids_covering_kml

# Example to generate a list of S2 cell IDs of a given level that cover the given kml file.

In [ ]:
s2_cell_level = 13
kml_file = 'https://raw.githubusercontent.com/datameet/Municipal_Spatial_Data/master/Chennai/CMA.kml'

polygons_vertices = fetch_polygons_in_kml(kml_file)
print(polygons_vertices)

##### Obtain S2 cells at specified level within the region #####
s2_cells_uint_64_format = get_s2_cells_covering_kml(polygons_vertices, s2_cell_level)

##### convert these cells to hex cells #####
s2_cells_hex_format = [hex(s2_cell) for s2_cell in s2_cells_uint_64_format]

print("s2_cells in uint64 format", s2_cells_uint_64_format)
print("s2_cells in hex format", s2_cells_hex_format)

[[(13.1093017901601, 80.3049883910702), (13.1087471626636, 80.3049507852118), (13.1078674915458, 80.3048173720021), (13.1071789028313, 80.3047796846587), (13.1068921977509, 80.3047222622106), (13.1067780150365, 80.3045674210725), (13.106720990401, 80.3044515577544), (13.1067221264446, 80.3041421131192), (13.1068950793961, 80.3039284674031), (13.1070292302327, 80.3038510130281), (13.1072208169134, 80.3037731315194), (13.1076032606834, 80.3038107857014), (13.1079663522102, 80.3038683337689), (13.1084638451683, 80.3038676216889), (13.1091527077809, 80.3038859091728), (13.1084085857775, 80.3034234594923), (13.1083138285734, 80.3032107513331), (13.1085248772199, 80.3030950712626), (13.109040058511, 80.3034030114732), (13.1098238555231, 80.3035561670215), (13.1117423384915, 80.302336137635), (13.1128743163168, 80.3015808998026), (13.1139048552774, 80.3021192749783), (13.1146536469832, 80.3014226525306), (13.116690756707, 80.2998263159196), (13.1158617205595, 80.3006861513203), (13.1139798941